## Il problema è di prevedere il prezzo di una casa date alcune sue caratteristiche.

Il vostro primo obiettivo è quello di arrivare a compiere l’operazione di previsione nel modo più rapido possibile: 
- non concentratevi sull’accuratezza o sul fare tutti gli step possibili nella fase di pulizia dei dati ma sull’avere un codice “minimo” che almeno funzioni. In base al tempo a vostra disposizione sceglierete eventualmente altri aspetti su cui concentrarvi.

La descrizione delle variabili del dataset è disponibile presso: https://www.kaggle.com/datasets/yasserh/housing-prices-dataset

In [ ]:
#--- Librerie
from pathlib import Path
import numpy as np                 # calcoli numerici, array
import pandas as pd                # DataFrame, manipolazione dati
import matplotlib.pyplot as plt    # grafici base
import seaborn as sns              # grafici statistici più belli
from sklearn.model_selection import train_test_split            # per dividere i dati in train e test
from sklearn.model_selection import cross_val_score             # valuta il modello con la cross-validation
from sklearn.linear_model import LinearRegression, Ridge, Lasso # modelli di regressione lineare, Ridge e Lasso
from sklearn.preprocessing import StandardScaler                # normalizza le caratteristiche (media 0, deviazione standard 1)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # le metriche per misurare quanto sbaglia il modello

# gestione percorso
root = Path.cwd()
file_path = root / 'Housing.csv'

In [ ]:
# Leggo il dataset
case_df = pd.read_csv(file_path)
case_df.info()

In [ ]:
# visualizzo prime 5 righe del dataset
case_df.head()

In [ ]:
# Mostra tutte le righe che sono considerate duplicati
is_duplicate = case_df[case_df.duplicated()]
print(is_duplicate)

In [ ]:
# Filtra il DataFrame mostrando solo le righe con almeno un NaN
is_null = case_df[case_df.isnull().any(axis=1)]
print(is_null)

*dopo una prima analisi:*
- non riscontro duplicati
- non riscontro valori null

## Encoding (nello specifico Binary Encoding) 
Procedo con trasformare variabili categoriche testuali in valori numerici in modo tale che il modello possa elaborarli.
- `mainroad`	
- `guestroom`	
- `basement`	
- `hotwaterheating`	
- `airconditioning`
- `prefarea`
- e in fine `furnishingstatus`

In [ ]:
cols_binarie = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
# Verifico e ottengo un set di tutti i valori presenti nelle colonne interessate
valori_unici = set(case_df[cols_binarie].values.ravel())
print(f"Valori presenti nelle colonne binarie: {valori_unici}\n")

# Verifica specifica per la categorica multipla della colonna 'furnishingstatus' uso value_counts()
# anche per capire se una categoria è rarissima
valori_cat_unici =case_df['furnishingstatus'].value_counts()
print(f"Valori presenti nella colonna 'furnishingstatus':\n{valori_cat_unici}")

In [ ]:
# provvediamo a fare gli encoding binari
mapping = {'yes': 1, 'no': 0}

for col in cols_binarie:
    case_df[col] = case_df[col].map(mapping)

# Verifica dell'encoding
print(case_df[cols_binarie].head())

In [ ]:
"""
# Trasformazione One-Hot Encoding (Variabili Dummy)
case_df = pd.get_dummies(case_df, columns=['furnishingstatus'], drop_first=True, dtype=int)

# Verifica le nuove colonne create
print(case_df.columns)

# Mostra le prime righe delle nuove colonne dummy
print(case_df.filter(like='furnishingstatus').head())
"""

In [ ]:
# Creiamo il dizionario di mappatura per la colonna 'furnishingstatus'
cat_mapping = {
    'unfurnished': 0,
    'semi-furnished': 1,
    'furnished': 2
}

# Applichiamo il mapping
case_df['furnishingstatus'] = case_df['furnishingstatus'].map(cat_mapping)

# Verifica del mapping
print("I valori contenuti nella colonna 'furnishingstatus' sono:",case_df['furnishingstatus'].unique())

In [ ]:
case_df.head()

### eseguo delle prime valutazioni
- distribuzione della variabile target (prezzo)
- quanto ogni caratteristica(features) è legata al prezzo

In [ ]:
# Analisi della distribuzione della variabile target
plt.figure(figsize=[8,4])
sns.histplot(case_df['price']/1e6, color='green', edgecolor="black", linewidth=2, bins=30, kde=True) # /1e6 equivale a 1 milione
plt.xlabel('Prezzo (Milioni di $)')
plt.ylabel('Frequenza (Numero di Case)')
plt.title('Distribuzione della Variabile Target - Valore Medio delle Case (in milioni di $)')
plt.show()

**Da questa distribuzione apprendo che:**
1. La colonna più alta si trova tra i 3 e i 4 milioni di $.
   - Il "cuore" del mercato immobiliare è concentrato su case di fascia medio-bassa. Se il nostro modello futuro dovesse prevedere un prezzo di 15 milioni per una casa normale, sapremo subito che c'è qualcosa che non va.
2. La coda del grafico si allunga verso destra.
   - il modello potrebbe essere influenzato eccessivamente dalle poche case di lusso, faticando a prevedere bene il prezzo delle case medie.
3. Presenza di Outliers.
   - le barre isolate oltre i 10 milioni (case "eccezionali" che costano molto più della media)
4. Il prezzo parte da circa 1.7 milioni e arriva a oltre 13 milioni (quasi 8 volte il prezzo minimo).
   - le caratteristiche devono essere molto forti per giustificare una differenza di prezzo così grande.

In [ ]:
# Correlation matrix — quanto ogni feature è legata al prezzo
# Una correlazione vicina a 1.0 significa che la feature cresce insieme al prezzo.
# Vicina a 0 significa che non influenza quasi nulla.
corr = case_df.corr()
print(corr["price"].sort_values(ascending=False))

# Heatmap visiva della correlation matrix
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Matrice di Correlazione delle caratteristiche")
plt.tight_layout()
plt.show()

### Preparazione dati

operiamo su due obiettivi:
1. dividere i dati in `training` e `test set`
2. normalizzare le features

> nota :  `Il test set` non va mai toccato prima della valutazione finale. Usarlo durante il training sarebbe come dare le risposte all'esame prima dell'interrogazione - il modello sembrerebbe bravo ma non lo sarebbe davvero. Questo problema si chiama data leakage.

> Nota importante: fit_transform() sul training, ma solo transform() sul test - mai fit_transform() sul test. Il motivo è che il test set deve essere scalato usando la media e la std calcolate sul training, non le sue. Altrimenti facciamo trapelare informazioni dal test set(data leakage)

In [ ]:
case_df = case_df[case_df['price'] <= 10_000_000]  # rimuovo outlier

X = case_df.drop(columns=["price"])  # tutto tranne il prezzo → features
y = case_df["price"]                 # solo il prezzo → target

print(X.shape)  # (545, 12) → 545 case, 12 feature
print(y.shape)  # (545,)   → prezzi

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # 20% dei dati per il test, 80% per il training
print(f"Training set : {X_train.shape[0]} campioni") # 436 campioni
print(f"Test set     : {X_test.shape[0]} campioni") # 109 campioni

In [ ]:
# Lo scaler trasforma ogni feature così:
# valore_scalato = (valore - media) / deviazione_standard
scaler = StandardScaler()

# fit_trasform sul trading → calcola media e STD DAI DATI DI TRAINING
X_train_scaled = scaler.fit_transform(X_train)

# transform sul test → usa media e STD CALCOLATI SUL TRAINING
X_test_scaled = scaler.transform(X_test)

In [ ]:
# --- CREAMO E ALLENIAMO I MODELLI

# istanze dei modelli:
lr = LinearRegression()
ridge = Ridge(alpha=1.0)  # alpha è il parametro di regolarizzazione
lasso = Lasso(alpha=500)  # alpha più alto perchè i prezzi sono molto grandi

# Training... il modello studia la relazione tra X e y
lr.fit(X_train_scaled, y_train)
ridge.fit(X_train_scaled, y_train)
lasso.fit(X_train_scaled, y_train)

In [ ]:
# --- Predizione sul test set:
y_pred_lr = lr.predict(X_test_scaled)
y_pred_ridge = ridge.predict(X_test_scaled)
y_pred_lasso = lasso.predict(X_test_scaled)
# Ora y_pred_lr contiene i 109 prezzi stimati dal modello, uno per ogni casa del test set.
# Dopo li confronteremo con i 109 prezzi reali in y_test

# Calcoliamo le metriche di valutazione per ogni modello
for name, y_pred in [("Linear Regression", y_pred_lr),
                    ("Ridge Regression", y_pred_ridge),
                    ("Lasso Regression", y_pred_lasso)]:

    # MAE (Mean Absolute Error): media degli errori assoluti tra i prezzi reali e quelli predetti.
    # Più è basso, meglio è.
    mae = mean_absolute_error(y_test, y_pred)

    # RMSE (Root Mean Squared Error): radice quadrata della media degli errori al quadrato.
    # Penalizza di più gli errori grandi. Anche qui, più è basso, meglio è.
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    # R² (R-squared): indica la proporzione della varianza nei prezzi reali.
    # 1.0 sarebbe predizione perfetta,
    # 0 significa equivale a predirre la media, s
    # e negativo significa che il modello è peggio di predirre la media.
    r2 = r2_score(y_test, y_pred)

    print(f"\n[{name}]")
    print(f"  MAE  : ${mae:,.0f}")
    print(f"  RMSE : ${rmse:,.0f}")
    print(f"  R²   : {r2:.4f}") # alt 0178 per ²

In [ ]:
for nome, modello in [("Linear Regression", lr),
                      ("Ridge (L2)",        ridge),
                      ("Lasso (L1)",        lasso)]:

    cv_scores = cross_val_score(
        modello,        # modello da valutare
        X_train_scaled, # Le feature di addestramento (scalate)
        y_train,        # target (la variabile che vogliamo predire)
        cv=5,           # divide il training in 5 parti
        scoring="r2"    # indica quanto bene il modello spiega la varianza dei dati
    )
    print(f"\n[{nome}]")

    # Stampiamo la media dei 5 punteggi e la deviazione standard
    # La media indica la performance generale, la deviazione standard indica
    # quanto il modello è "instabile" o sensibile ai dati specifici
    print(f"  CV R² : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    # Stampiamo i singoli punteggi ottenuti in ciascuna delle 5 iterazioni
    print(f"  Scores: {cv_scores.round(4)}")

In [ ]:
# --- Grafici Comparativi (Reale vs Predetto)
# Ogni punto è una casa:
#   sull'asse X il prezzo reale,
#   sull'asse Y quello predetto dal modello.

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Valori Reali vs Predetti — Confronto Modelli", fontsize=14)

dati = [
    ("Linear Regression", y_pred_lr,     0.6980), # R² calcolato in precedenza
    ("Ridge (L2)",         y_pred_ridge, 0.6978),
    ("Lasso (L1)",         y_pred_lasso, 0.6978),
]

for ax, (nome, y_pred, r2) in zip(axes, dati):
    ax.scatter(y_test, y_pred, alpha=0.6, edgecolors="k", linewidths=0.3)

    # Linea rossa tratteggiata = predizione perfetta
    lim = [y_test.min(), y_test.max()]
    ax.plot(lim, lim, "r--", lw=2, label="Perfetto")

    ax.set_xlabel("Prezzo Reale (€)")
    ax.set_ylabel("Prezzo Predetto (€)")
    ax.set_title(f"{nome}\nR²={r2:.4f}")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- dopo l'addestramento, possiamo guardare i coefficienti del modello lineare per capire quali feature hanno più impatto sul prezzo.

coef_df = pd.DataFrame({
    "feature"     : X.columns,         # nomi delle colonne
    "coefficiente": lr.coef_.round(2)  # pesi appresi dal modello
}).sort_values("coefficiente", ascending=False)

print(coef_df.to_string(index=False))
# ⚠️ Attenzione — i dati sono stati **scalati** con StandardScaler, quindi i coefficienti NON sono in $ diretti.
# Rappresentano l'impatto sul prezzo per di quella feature.

# bagno            → feature più importante, impatto maggiore sul prezzo
# camera da letto  → feature meno importante, impatto minore

In [ ]:
plt.figure(figsize=(9, 5))

# Verde se il coefficiente è positivo, rosso se negativo
colors = ["#2ecc71" if c > 0 else "#e74c3c" for c in coef_df["coefficiente"]]

plt.barh(coef_df["feature"], coef_df["coefficiente"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)  # linea verticale a 0
plt.title("Feature Importance — Coefficienti (scalati)", fontsize=13)
plt.xlabel("Valore del Coefficiente")
plt.tight_layout()
plt.show()

### Predizione su una Casa di Esempio
Creamo una casa con caratteristiche specifiche e chiediamo ai 3 modelli quanto vale.

In [ ]:
nuova_casa = pd.DataFrame([{
    'area' : 2500,
    'bedrooms' : 3,
    'bathrooms' : 3,
    'stories' : 2,
    'mainroad' : 1,
    'guestroom' : 1,
    'furnishingstatus' : 0, # unfurnished
    'basement' : 1,
    'hotwaterheating' : 1,
    'airconditioning' : 1,
    'parking' : 2,
    'prefarea' : 1
}])

# ricevevo errore perché le colonne di nuova_casa erano in ordine diverso rispetto a quelle di X_train
ordina_colonne = X_train.columns
nuova_casa = nuova_casa[ordina_colonne]

# ⚠️ IMPORTANTE: usiamo .transform() e NON .fit_transform()
# Lo scaler è già stato addestrato sul training set — dobbiamo usare la stessa scala, non ricalcolarla su questa singola casa
nuova_casa_sc = scaler.transform(nuova_casa)

for nome, modello in [("Linear Regression",  lr),
                      ("Ridge (L2)",         ridge),
                      ("Lasso (L1)",         lasso)]:
    prezzo = modello.predict(nuova_casa_sc)[0]  # [0] perché predict restituisce un array
    print(f"{nome:<22} → $ {prezzo:,.0f}")

In [ ]:
# Verifica manuale del prezzo
# prezzo = intercetta + (coeff_0 × x_0) + (coeff_1 × x_1) + ... + (coeff_n × x_n)
# prezzo = intercetta + somma(coeff_i * val_scalato_i)

# lr.intercept_ → è il punto di partenza del modello (la costante)
# lr.coef_[i] → è il peso che il modello ha assegnato a ogni feature
# nuova_casa_sc[0][i] → è il valore scalato di ogni feature della nostra casa
# sum(...) → somma tutti i prodotti coeff × valore

prezzo_manuale = lr.intercept_ + sum(
    lr.coef_[i] * nuova_casa_sc[0][i]
    for i in range(len(lr.coef_))
)

print(f"Prezzo manuale : $ {prezzo_manuale:,.0f}")
print(f"Prezzo predict : $ {lr.predict(nuova_casa_sc)[0]:,.0f}")

differenza = abs(prezzo_manuale - lr.predict(nuova_casa_sc)[0])
if differenza < 1:
    print("✅ La predizione è corretta!")
else:
    print(f"❌ Qualcosa non torna, differenza: $ {differenza:,.2f}")


In [ ]:
# --- Calcoliamo i residui (errori) della predizione per ogni casa del test set
residui = y_test - y_pred_lr

print(f"Media residui : €{residui.mean():,.0f}")   # ideale → 0
print(f"Std residui   : €{residui.std():,.0f}")
print(f"Min residuo   : €{residui.min():,.0f}")
print(f"Max residuo   : €{residui.max():,.0f}")